In [ ]:
# Open AI API key 등록
import os

OPENAI_API_KEY='API_KEY'


In [ ]:
# 현재 노트북 커널에 환경변수 등록
os.environ['OPENAI_API_KEY']=OPENAI_API_KEY

In [ ]:
# 랭스미스로 기록 할 건지 여부
LANGSMITH_TRACING="true"

# 랭스미스 사이트에서 기록 할 나만의 url
LANGSMITH_ENDPOINT="https://api.smith.langchain.com"

# 내 랭스미스 url key
LANGSMITH_API_KEY="API_KEY"

# 내 계정의 프로젝트이름
LANGSMITH_PROJECT="langchain0422"

# 현재 노트북 커널에 환경변수 등록
os.environ['LANGSMITH_TRACING']=LANGSMITH_TRACING
os.environ['LANGSMITH_ENDPOINT']=LANGSMITH_ENDPOINT
os.environ['LANGSMITH_API_KEY']=LANGSMITH_API_KEY
os.environ['LANGSMITH_PROJECT']=LANGSMITH_PROJECT

In [ ]:
# LangChain 라이브러리 설치하기

!pip install langchain langchain-openai langchain-community faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.7 MB/s eta 0:00:00


### 제로샷 프롬프팅 (Zero-Shot Prompting)
- Zero-Shot : AI 모델이 사전 학습 없이 새로운 작업이나 상황에 대응하는 방식
- Zero-Shot Prompting : **사전 예시 없이**, 단순히 요청만 주어져 모델이 답변을 생성하는 방식

> **특징**  
> - **<font color=red>추가 예시 없이</font>** 문제 설명만으로 작업을 수행함  
> - 모델은 오로지 프롬프트에 포함된 정보만을 기반으로 응답을 생성함  
> - **장점** : `빠르고 간단하게 적용할 수 있음`  
> - **단점** : `복잡한 문제나 구체적인 형식을 요구하는 작업에서는 모델의 응답이 기대에 못 미칠 수 있음`

### Zero-Shot Prompting 개선 방법
- Few-Shot Prompting
- 메모리 기능 사용
- 체인 결합 (여러개의 언어 모델 연결)
- RAG (검색, 증강, 생성)
- TAG (테이블(DB), 증강, 생성)
- Agent 등

### Few-Shot Promptiong(퓨샷프롬프팅)
- LLM 모델에 넣는 프롬프트에 예시 텍스트를 넣어서 새로운 작업에 대한 답변의 환각을 줄이는 방법

In [ ]:
# 퓨샷에 필요한 프롬프트 라이브러리

from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate    # 퓨샷프롬프팅을 위한 템플릿
from langchain.chat_models import init_chat_model                           # 유연하게 다양한 LLM을 불러오는 함수
from langchain_core.output_parsers import StrOutputParser                   # 문자열 파서


In [ ]:
# 모델생성

llm = init_chat_model(model='openai:gpt-4o-mini', max_tokens=1024)

In [ ]:
# 예시데이터셋 (input의 반댓말을 output으로 출력하는 예시)

example1 = [
    {"input":"행복", "output":"불행"},
    {"input":"여름", "output":"겨울"},
    {"input":"아침", "output":"저녁"},
    {"input":"달콤한", "output":"쓴"},
    {"input":"배고프다", "output":"배부르다"},
    {"input":"건조한", "output":"습한"},
]

In [ ]:
# 예시 프롬프트를 생성하는 템플릿

example_template = PromptTemplate.from_template("입력단어: {input}\n출력단어 : {output}\n")

In [ ]:
# 테스트 하기

print(example_template.invoke(example1[0]).text)

입력단어: 행복
출력단어 : 불행



In [ ]:
# 전체 데이터에 적용해주는 FewShot Template 생성

fewshot_template = FewShotPromptTemplate(
    examples=example1,                # 예시데이터 연결하기.
    example_prompt=example_template,  # 예시데이터 템플릿 연결하기.

    # 예시데이터 윗쪽 지시문
    prefix="아래 데이터를 참고해 입력의 반댓말을 출력한다. 예시데이터 출력형식을 지킬 것",
    # 예시데이터 아래쪽 지시문
    suffix="입력단어: {input_keyword}",

    input_variables=['input_keyword']  # 입력변수 지정
)

In [ ]:
# 테스트

print(fewshot_template.invoke('웃다').text)

아래 데이터를 참고해 입력의 반댓말을 출력한다. 예시데이터 출력형식을 지킬 것

입력단어: 행복
출력단어 : 불행


입력단어: 여름
출력단어 : 겨울


입력단어: 아침
출력단어 : 저녁


입력단어: 달콤한
출력단어 : 쓴


입력단어: 배고프다
출력단어 : 배부르다


입력단어: 건조한
출력단어 : 습한


입력단어: 웃다


In [ ]:
# 체인구성하기

chain = fewshot_template | llm | StrOutputParser()
chain.invoke('웃다')

'출력단어 : 울다'

In [ ]:
# 여러 데이터 사용하기 (batch사용)

chain.batch(["맛없는","못생긴","상큼한"])

['출력단어 : 맛있는', '출력단어 : 잘생긴', '출력단어 : 시큼한']

### 복잡한 질문에 대해서 단계적으로 답변 구성하기

In [ ]:
# 예시 데이터 생성: 복잡한 질문에 대해 단계별로 답변을 구성하는 예시
# 테마 : 라이브 커머스 대본 자동 생성 서비스

examples2 = [
    {
        "question": "상품명: '에어프라이어', 특징: '간편 조리, 1300W 출력', 가격: '120,000원'일 때 단계별 대본 가이드 예시를 보여주세요.",
        "answer": (
            "1) 인사 및 상품 소개: \"안녕하세요 여러분! 오늘은 집에서 간편하게 튀김 요리를 할 수 있는 '에어프라이어'를 소개합니다!\"\n"
            "2) 주요 특징 설명: \"1300W의 강력한 출력으로 바삭함을 살려주고, 조리 시간이 단축됩니다.\"\n"
            "3) 사용 시나리오 제시: \"감자튀김, 치킨너겟, 채소구이까지 모두 간편하게 조리해보세요.\"\n"
            "4) 가격 및 혜택 안내: \"지금 구매하시면 120,000원에 무료 배송과 1년 무상 A/S가 제공됩니다.\"\n"
            "5) 콜 투 액션: \"아래 구매 버튼을 눌러 할인 혜택을 놓치지 마세요!\""
        )
    },
    {
        "question": "상품명: '블루투스 헤드폰', 특징: '노이즈 캔슬링, 20시간 배터리', 가격: '80,000원'일 때 단계별 대본 가이드 예시를 보여주세요.",
        "answer": (
            "1) 인사 및 관심 유도: \"안녕하세요! 오늘은 음악 감상에 최적화된 '블루투스 헤드폰'을 소개드립니다.\"\n"
            "2) 기능 강조: \"능동형 노이즈 캔슬링으로 외부 소음을 완벽 차단, 최대 20시간 배터리로 하루 종일 사용 가능해요.\"\n"
            "3) 데모 시연 제안: \"실제 사용해볼까요? 소음 많은 카페에서도 깨끗한 사운드를 경험해보세요.\"\n"
            "4) 특별 할인 안내: \"오늘 한정 5% 할인 쿠폰 코드: LIVE5 적용하세요!\"\n"
            "5) 구매 유도 문구: \"지금 바로 구매 버튼을 클릭하고, 편안한 음악 시간을 즐겨보세요!\""
        )
    }
]

In [ ]:
# 예시 변환용 템플릿

example_template = PromptTemplate.from_template(
    "상품정보: {question}\n 라이브커머스 대본: \n{answer}"
)

In [ ]:
# 테스트

print(example_template.invoke(examples2[0]).text)

상품정보: 상품명: '에어프라이어', 특징: '간편 조리, 1300W 출력', 가격: '120,000원'일 때 단계별 대본 가이드 예시를 보여주세요.
 라이브커머스 대본: 
1) 인사 및 상품 소개: "안녕하세요 여러분! 오늘은 집에서 간편하게 튀김 요리를 할 수 있는 '에어프라이어'를 소개합니다!"
2) 주요 특징 설명: "1300W의 강력한 출력으로 바삭함을 살려주고, 조리 시간이 단축됩니다."
3) 사용 시나리오 제시: "감자튀김, 치킨너겟, 채소구이까지 모두 간편하게 조리해보세요."
4) 가격 및 혜택 안내: "지금 구매하시면 120,000원에 무료 배송과 1년 무상 A/S가 제공됩니다."
5) 콜 투 액션: "아래 구매 버튼을 눌러 할인 혜택을 놓치지 마세요!"


In [ ]:
# 퓨샷프롬프트 만들기

fewshot_template = FewShotPromptTemplate(
    examples=examples2,               # 예시데이터 연결
    example_prompt=example_template,# 예시용 템플릿 연결
    prefix="아래 예시를 참고하여 라이브커머스 대본을 생성한다.",
    suffix="상품정보: {info}",
    input_variables=['info']
)

In [ ]:
# 테스트용
sample_info = "상품명: 고구마프라이, 특징: 바삭하고 달콤한 맛, 가격: 3200원"

In [ ]:
print(fewshot_template.invoke(sample_info).text)

아래 예시를 참고하여 라이브커머스 대본을 생성한다.

상품정보: 상품명: '에어프라이어', 특징: '간편 조리, 1300W 출력', 가격: '120,000원'일 때 단계별 대본 가이드 예시를 보여주세요.
 라이브커머스 대본: 
1) 인사 및 상품 소개: "안녕하세요 여러분! 오늘은 집에서 간편하게 튀김 요리를 할 수 있는 '에어프라이어'를 소개합니다!"
2) 주요 특징 설명: "1300W의 강력한 출력으로 바삭함을 살려주고, 조리 시간이 단축됩니다."
3) 사용 시나리오 제시: "감자튀김, 치킨너겟, 채소구이까지 모두 간편하게 조리해보세요."
4) 가격 및 혜택 안내: "지금 구매하시면 120,000원에 무료 배송과 1년 무상 A/S가 제공됩니다."
5) 콜 투 액션: "아래 구매 버튼을 눌러 할인 혜택을 놓치지 마세요!"

상품정보: 상품명: '블루투스 헤드폰', 특징: '노이즈 캔슬링, 20시간 배터리', 가격: '80,000원'일 때 단계별 대본 가이드 예시를 보여주세요.
 라이브커머스 대본: 
1) 인사 및 관심 유도: "안녕하세요! 오늘은 음악 감상에 최적화된 '블루투스 헤드폰'을 소개드립니다."
2) 기능 강조: "능동형 노이즈 캔슬링으로 외부 소음을 완벽 차단, 최대 20시간 배터리로 하루 종일 사용 가능해요."
3) 데모 시연 제안: "실제 사용해볼까요? 소음 많은 카페에서도 깨끗한 사운드를 경험해보세요."
4) 특별 할인 안내: "오늘 한정 5% 할인 쿠폰 코드: LIVE5 적용하세요!"
5) 구매 유도 문구: "지금 바로 구매 버튼을 클릭하고, 편안한 음악 시간을 즐겨보세요!"

상품정보: 상품명: 고구마프라이, 특징: 바삭하고 달콤한 맛, 가격: 3200원


In [ ]:
# 체인구성 및 호출

chain = fewshot_template | llm | StrOutputParser()
print(chain.invoke(sample_info))

라이브커머스 대본:

1) 인사 및 상품 소개: "안녕하세요 여러분! 오늘은 간편하게 즐길 수 있는 맛있는 스낵, '고구마프라이'를 소개합니다!"

2) 주요 특징 설명: "바삭하고 달콤한 맛이 일품인 고구마프라이는 누구나 좋아하는 간식이에요."

3) 사용 시나리오 제시: "혼자 즐기기 좋은 영양 간식으로도, 가족이나 친구들과 함께 나누기에도 딱이죠!"

4) 가격 안내: "그리고 가격은 단 3,200원으로, 부담 없이 즐길 수 있는 가격입니다!"

5) 콜 투 액션: "지금 바로 아래 구매 버튼을 눌러서 바삭하고 달콤한 고구마프라이를 놓치지 마세요!"


- 문제점: 단순 FewShot Prompting은 예시데이터가 많아지거나 하나의 예시가 길어지면 token 사용량이 올라간다.

### ExampleSelector
- 사용자 질의와 연관있는 예시를 골라서 퓨샷프롬프팅하는 방법

- 예시가 많으면 토큰 사용량이 많아지고 따라서 API 사용료가 많이 나옴
- 최종 질문과 유사한 예시 한 두 개만 프롬프트에 포함할 예제를 선택하는 경우에 사용

- 종류
  - <font color=red>SemanticSimilarityExampleSelector</font>: 벡터 임베딩의 코사인 유사도를 이용해 질문과 의미가 가까운 예시를 선택
---
- 상황에 따라 사용
  - <font color=red>BaseExampleSelector</font>: 상속 받아 사용자 정의 ExampleSelector를 생성
  - <font color=red>LengthBasedExampleSelector</font>: 지정 길이가 넘어가지 않도록 예시 개수를 조절
  - <font color=red>MaxMarginalRelevanceExampleSelector</font>: Maximal Marginal Relevance (MMR)을 이용해 질문과 가까우면서도 다양한 예시를 선택
  - <font color=red>NGramOverlapExampleSelector</font>: n-gram overlap score를 이용해 질문과 가까운 예시를 선택


In [ ]:
# 유사도를 비교해 예시를 골라주는 라이브러리
from langchain_core.example_selectors import SemanticSimilarityExampleSelector

# 임베딩 모델 (글자 -> 숫자)
from langchain_openai.embeddings import OpenAIEmbeddings

# 벡터데이터베이스
from langchain_community.vectorstores import FAISS

In [ ]:
# 첫번째 예시는 회의록 (회의에서 진행되었던 내용 기반으로 내용에 대해 나열)
# 두번째 요소는 요약문 (문장의 핵심적인 부분을 추출하는 Action)

examples3 = [
    {
        "input": (
            "2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다. "
            "회의에는 마케팅 팀장인 김수진, 디지털 마케팅 담당자인 박지민, 소셜 미디어 관리자인 이준호가 참석했다. "
            "회의의 주요 목적은 2024년 상반기 마케팅 전략을 수립하고, 새로운 소셜 미디어 캠페인에 대한 아이디어를 논의하는 것이었다. "
            "팀장인 김수진은 최근 시장 동향에 대한 간략한 개요를 제공했으며, 이어서 각 팀원이 자신의 분야에서의 전략적 아이디어를 발표했다."
        ),
        "answer": (
            "회의록: XYZ 회사 마케팅 전략 회의\n"
            "일시: 2023년 12월 25일\n"
            "장소: XYZ 회사 회의실\n"
            "참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)\n\n"
            "1. 개회\n"
            "   - 김수진 팀장의 개회사\n"
            "2. 시장 동향 개요 (김수진)\n"
            "   - 최근 시장 동향 분석\n"
            "3. 디지털 마케팅 전략 (박지민)\n"
            "   - SEO 최적화 및 온라인 광고 방안\n"
            "4. 소셜 미디어 캠페인 (이준호)\n"
            "   - 인플루언서 마케팅 제안\n"
            "5. 종합 논의\n"
            "   - 예산 및 자원 배분\n"
            "6. 마무리\n"
            "   - 다음 회의 일정 및 회의록 배포 담당자 지정"
        )
    },
    {
        "input": (
            "이 문서는 '지속 가능한 도시 개발을 위한 전략'에 대한 20페이지 분량의 보고서입니다. "
            "보고서는 지속 가능한 도시 개발의 중요성, 현재 도시화의 문제점, 그리고 지속 가능한 개발 전략을 다루고 있습니다. "
            "여러 국가의 성공 사례와 얻은 교훈도 포함되어 있습니다."
        ),
        "answer": (
            "문서 요약: 지속 가능한 도시 개발 전략 보고서\n\n"
            "- 중요성: 사회적·경제적·환경적 이점 강조\n"
            "- 문제점: 환경 오염, 자원 고갈, 불평등 분석\n"
            "- 전략: 친환경 건축, 대중교통 개선, 에너지 효율성 증대\n"
            "- 사례 연구: 코펜하겐, 요코하마 성공 사례\n"
            "- 교훈: 다각적 접근, 지역사회 협력, 장기 계획 필요"
        )
    }
]

In [ ]:
# 예시용 프롬프트 템플릿

example_template = PromptTemplate.from_template("원문: \n{input}\n\n변환결과: \n{answer}")

In [ ]:
# 테스트

print(example_template.invoke(examples3[0]).text)

원문: 
2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다. 회의에는 마케팅 팀장인 김수진, 디지털 마케팅 담당자인 박지민, 소셜 미디어 관리자인 이준호가 참석했다. 회의의 주요 목적은 2024년 상반기 마케팅 전략을 수립하고, 새로운 소셜 미디어 캠페인에 대한 아이디어를 논의하는 것이었다. 팀장인 김수진은 최근 시장 동향에 대한 간략한 개요를 제공했으며, 이어서 각 팀원이 자신의 분야에서의 전략적 아이디어를 발표했다.

변환결과: 
회의록: XYZ 회사 마케팅 전략 회의
일시: 2023년 12월 25일
장소: XYZ 회사 회의실
참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)

1. 개회
   - 김수진 팀장의 개회사
2. 시장 동향 개요 (김수진)
   - 최근 시장 동향 분석
3. 디지털 마케팅 전략 (박지민)
   - SEO 최적화 및 온라인 광고 방안
4. 소셜 미디어 캠페인 (이준호)
   - 인플루언서 마케팅 제안
5. 종합 논의
   - 예산 및 자원 배분
6. 마무리
   - 다음 회의 일정 및 회의록 배포 담당자 지정


In [ ]:
# 예시를 골라주는 example selector 생성

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples  =examples3,           # 유사도 비교할 예시데이터 연결
    embeddings=OpenAIEmbeddings(),  # 임베딩 도구 연결
    vectorstore_cls=FAISS,          # 벡터데이터베이스 연결
    k=1                             # 선택할 유사 데이터 갯수 선택
)

In [ ]:
# 퓨샷프롬프트 템플릿 생성

fewshot_prompt=FewShotPromptTemplate(
    example_selector= example_selector,  # 예시 선택도구 연결
    example_prompt  = example_template,  # 골라진 예시데이터를 프롬프트로 변경하는 템플릿 연결

    # 지시문
    prefix="아래 예시를 참고하여 입력된 텍스트를 적절한 형식으로 변환하시오.",
    suffix="원문: \n{document}",
    input_variables=["document"]
)

In [ ]:
# 체인 구성 및 호출

chain = fewshot_prompt | llm | StrOutputParser()

In [ ]:
# 회의록버전

print(chain.invoke({"document" : (
        "2025년 5월 20일 오후 4시부터 약 90분간, 서울 본사 3층 대회의실에서 "
        "신제품 출시 킥오프 회의가 진행되었습니다. 회의에는 개발팀장 김민준, "
        "디자인팀장 이지은, 영업팀장 박성호, 마케팅팀장 오세훈, 품질관리 담당 최유리가 참석했으며, "
        "주요 안건은 제품 디자인 확정, 출시 일정, 마케팅 채널 전략, 리스크 관리 방안이었습니다."
    )}))

회의록: 신제품 출시 킥오프 회의  
일시: 2025년 5월 20일  
장소: 서울 본사 3층 대회의실  
참석자: 김민준 (개발팀장), 이지은 (디자인팀장), 박성호 (영업팀장), 오세훈 (마케팅팀장), 최유리 (품질관리 담당)

1. 개회
   - 김민준 팀장의 개회사
2. 제품 디자인 확정 (이지은)
   - 최종 디자인안 발표
3. 출시 일정 (오세훈)
   - 제품 출시 일정 논의
4. 마케팅 채널 전략 (오세훈)
   - 광고 및 프로모션 계획
5. 리스크 관리 방안 (최유리)
   - 잠재적 위험 요소 및 대응 방안 논의
6. 종합 논의
   - 각 부서 간 협력 방안 및 일정 조율
7. 마무리
   - 다음 회의 일정 및 회의록 배포 담당자 지정


In [ ]:
#
print(chain.invoke({"document" : (
        "이 문서는 '머신러닝 모델 성능 최적화 기법'에 대한 15페이지 분량의 기술 보고서입니다. "
        "보고서는 하이퍼파라미터 튜닝(그리드 서치, 베이지안 최적화), 데이터 증강(회전·크롭·색상 보정), "
        "앙상블 기법(배깅·부스팅·스태킹), 모델 경량화(지식 증류·양자화), 성능 평가(Accuracy·Precision·Recall·F1) 등을 상세히 다룹니다."
    )}))


문서 요약: 머신러닝 모델 성능 최적화 기법 보고서

- 하이퍼파라미터 튜닝: 그리드 서치, 베이지안 최적화 설명
- 데이터 증강: 회전, 크롭, 색상 보정 기법 소개
- 앙상블 기법: 배깅, 부스팅, 스태킹 방법론 상세
- 모델 경량화: 지식 증류, 양자화 기법 설명
- 성능 평가: Accuracy, Precision, Recall, F1 지표 분석
